# WFI2033 BIC comparison

This notebook compares three lens-model runs with BIC-like scores: Step 5 EPL, Step 6 composite with Pantheon+SH0ES, and Step 6 composite with DESI+Planck. It deliberately separates the comparison into three observable likelihood choices:

1. Image only: BIC = χ²_image,best + k ln N_pixel.
2. Image + measured time delays: BIC = χ²_image+TD,best + k ln (N_pixel + 3). A single best-fit time-delay scale is fitted for each posterior sample, so this comparison does not directly compare cosmology priors.
3. Image + measured time delays + κ_ext constraint: BIC = χ²_image+TD+κ,best + k ln (N_pixel + 4).

The calculation uses the same image mask and noise map for the image-plane term. For Step 6, the image is the lens-light-subtracted image and the stored model is the source+point-source model. This is equivalent to comparing the full image after adding the fixed Step 5 lens-light model, up to the fixed subtraction convention.

Important: the time-delay and full-observable scores require fpd_31, fpd_32, and fpd_34 in each posterior file. The Step 6 chains already store these variables. For the Step 5 EPL chain, use the fpd-enriched posterior generated by the Fermat-potential block in Step_5_data_visullization.ipynb, or save post_with_fpd and point EPL_NC to that file.


In [ ]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from astropy.io import fits
from tqdm.auto import tqdm


In [ ]:
# ============================================================
# Paths
# ============================================================
ROOT = Path('.').resolve()
LUSTRE_ROOT = Path('/mnt/lustre/tianli/quasar_hmc')
PAPER_DIR = ROOT / 'Paper'
PAPER_DIR.mkdir(exist_ok=True)

STEP5_SUFFIX = '_ss=2_fullconcen_light_multimass'
STEP5_RUN_TAG = '20260427_14'
STEP5_RESULT_DIR = ROOT / 'result' / f'result{STEP5_SUFFIX}_{STEP5_RUN_TAG}'
STEP5_DATA_PRODUCTS = STEP5_RESULT_DIR / 'data_products'

EPL_NC = LUSTRE_ROOT / f'WFI2033{STEP5_SUFFIX}_{STEP5_RUN_TAG}' / f'WFI2033_all{STEP5_SUFFIX}.nc'
SH0ES_NC = LUSTRE_ROOT / 'WFI2033_ss=2_fullconcen_light_multimass_step6_pantheonsh0es_reuse_redwarmup_20260507_14' / 'WFI2033_all_ss=2_fullconcen_light_multimass_step6_pantheonsh0es_reuse_redwarmup.nc'
DESI_PLANCK_NC = LUSTRE_ROOT / 'WFI2033_ss=2_fullconcen_light_multimass_step6_desi_planck_reuse_redwarmup_20260522_15' / 'WFI2033_all_ss=2_fullconcen_light_multimass_step6_desi_planck_reuse_redwarmup.nc'

FULL_DATA_PATH = STEP5_DATA_PRODUCTS / f'data_bkg_sub{STEP5_SUFFIX}.fits'
DATA_MINUS_LENS_LIGHT_PATH = STEP5_RESULT_DIR / f'data_minus_lens_light_corrected_psf{STEP5_SUFFIX}.fits'
RMS_PATH = STEP5_DATA_PRODUCTS / f'rms_with_psf_extra{STEP5_SUFFIX}.fits'
MASK_PATH = STEP5_DATA_PRODUCTS / f'mask_out{STEP5_SUFFIX}.fits'

print('EPL_NC =', EPL_NC)
print('SH0ES_NC =', SH0ES_NC)
print('DESI_PLANCK_NC =', DESI_PLANCK_NC)
print('FULL_DATA_PATH =', FULL_DATA_PATH)
print('DATA_MINUS_LENS_LIGHT_PATH =', DATA_MINUS_LENS_LIGHT_PATH)


In [ ]:
# ============================================================
# Model configuration
# ============================================================
# These variables are counted as active parameters for the BIC penalty.
# Edit these lists if you want lens-parameter-only BIC instead of full nuisance-parameter BIC.
EPL_FREE_VARS = [
    'theta_E_1', 'gamma_1', 'e_1', 'center_1', 'gamma_sheer_1',
    'theta_E_g1', 'theta_E_g2',
    'ra_ps', 'dec_ps', 'log10_amp_ps',
    'A_lens', 'amp_lens', 'sigma_lens', 'e_lens', 'center_lens',
    'n_source_grid', 'rho_source_grid', 'sigma_source_grid', 'pixels_wn_source_grid',
    'log_psf_corr_center',
]

COMPOSITE_FREE_VARS = [
    'm2l_ratio', 'm2l_ratio_slope',
    'kappa_s_halo', 'gammain_halo', 'e_halo', 'Rs_halo', 'center_halo', 'gamma_sheer_halo',
    'kappa_ext',
    'theta_E_g1', 'theta_E_g2',
    'ra_ps', 'dec_ps', 'log10_amp_ps',
    'n_source_grid', 'rho_source_grid', 'sigma_source_grid', 'pixels_wn_source_grid',
]

@dataclass
class ModelConfig:
    name: str
    nc_path: Path
    data_path: Path
    rms_path: Path
    mask_path: Path
    free_vars: list
    fixed_kappa_ext: float | None = None
    manual_k_offset: int = 0

MODEL_CONFIGS = [
    ModelConfig('EPL', EPL_NC, FULL_DATA_PATH, RMS_PATH, MASK_PATH, EPL_FREE_VARS, fixed_kappa_ext=0.0),
    ModelConfig('Composite Pantheon+SH0ES', SH0ES_NC, DATA_MINUS_LENS_LIGHT_PATH, RMS_PATH, MASK_PATH, COMPOSITE_FREE_VARS),
    ModelConfig('Composite DESI+Planck', DESI_PLANCK_NC, DATA_MINUS_LENS_LIGHT_PATH, RMS_PATH, MASK_PATH, COMPOSITE_FREE_VARS),
]

# Measured time delays in the image ordering used by the HMC scripts:
# fpd_31 -> B-A1, fpd_32 -> B-A2, fpd_34 -> B-C.
TIME_DELAY_OBS = {
    'fpd_31': {'label': 'B-A1', 'mean': -36.2, 'sigma_minus': 2.3, 'sigma_plus': 1.6},
    'fpd_32': {'label': 'B-A2', 'mean': -37.3, 'sigma_minus': 3.0, 'sigma_plus': 2.6},
    'fpd_34': {'label': 'B-C',  'mean': -59.4, 'sigma_minus': 1.3, 'sigma_plus': 1.3},
}

KAPPA_EXT_OBS = {'mean': 0.059, 'sigma_minus': 0.047, 'sigma_plus': 0.077}
DRAW_CHUNK = 200


In [ ]:
# ============================================================
# Helpers
# ============================================================
def read_fits_array(path, dtype=float):
    return np.asarray(fits.getdata(path), dtype=dtype)

def scalar_da(da):
    extra_dims = [d for d in da.dims if d not in ('chain', 'draw')]
    if extra_dims:
        da = da.isel({d: 0 for d in extra_dims})
    return da

def split_chi2(value, mean, sigma_minus, sigma_plus):
    resid = value - mean
    sigma = np.where(resid < 0.0, sigma_minus, sigma_plus)
    return (resid / sigma) ** 2

def count_parameter_size(ds, var_name):
    if var_name not in ds:
        raise KeyError(f'{var_name} missing from posterior')
    da = ds[var_name]
    size = 1
    for dim in da.dims:
        if dim not in ('chain', 'draw'):
            size *= int(da.sizes[dim])
    return int(size)

def count_model_parameters(nc_path, free_vars, manual_k_offset=0):
    ds = xr.open_dataset(nc_path, group='posterior', engine='h5netcdf')
    rows = []
    total = int(manual_k_offset)
    try:
        for name in free_vars:
            n = count_parameter_size(ds, name)
            rows.append({'var': name, 'n_param': n})
            total += n
    finally:
        ds.close()
    return total, pd.DataFrame(rows)

def compute_image_chi2(cfg, draw_chunk=DRAW_CHUNK):
    data = read_fits_array(cfg.data_path, dtype=float)
    rms = read_fits_array(cfg.rms_path, dtype=float)
    mask = read_fits_array(cfg.mask_path, dtype=bool)
    data_vec = data[mask]
    rms_vec = rms[mask]
    n_pix = int(mask.sum())

    ds = xr.open_dataset(cfg.nc_path, group='posterior', engine='h5netcdf')
    n_chain = int(ds.sizes['chain'])
    n_draw = int(ds.sizes['draw'])
    chi2 = np.empty((n_chain, n_draw), dtype=np.float64)
    try:
        for c in tqdm(range(n_chain), desc=f'{cfg.name}: image χ²'):
            for start in range(0, n_draw, draw_chunk):
                stop = min(start + draw_chunk, n_draw)
                model = ds['model_image'].isel(chain=c, draw=slice(start, stop)).load().values
                model_vec = model[:, mask]
                chi2[c, start:stop] = np.sum(((data_vec[None, :] - model_vec) / rms_vec[None, :]) ** 2, axis=1)
    finally:
        chain_coord = np.asarray(ds['chain'].values)
        draw_coord = np.asarray(ds['draw'].values)
        ds.close()

    return xr.DataArray(chi2, dims=('chain', 'draw'), coords={'chain': chain_coord, 'draw': draw_coord}), n_pix

def load_small_posterior(nc_path, names):
    ds = xr.open_dataset(nc_path, group='posterior', engine='h5netcdf')
    try:
        out = ds[names].load()
    finally:
        ds.close()
    return out


In [ ]:
# ============================================================
# Time-delay and κ_ext likelihood terms
# ============================================================
def time_delay_chi2_with_best_scale(post):
    fpd_names = list(TIME_DELAY_OBS.keys())
    missing = [name for name in fpd_names if name not in post]
    if missing:
        raise KeyError(f'Missing Fermat-potential variables: {missing}. Add fpd_31, fpd_32, fpd_34 to this chain first.')

    fpd = np.stack([scalar_da(post[name]).values for name in fpd_names], axis=-1)
    dt_mean = np.array([TIME_DELAY_OBS[name]['mean'] for name in fpd_names], dtype=float)
    sigma_minus = np.array([TIME_DELAY_OBS[name]['sigma_minus'] for name in fpd_names], dtype=float)
    sigma_plus = np.array([TIME_DELAY_OBS[name]['sigma_plus'] for name in fpd_names], dtype=float)
    sigma_sym = 0.5 * (sigma_minus + sigma_plus)

    weight = 1.0 / sigma_sym**2
    numerator = np.sum(fpd * dt_mean[None, None, :] * weight[None, None, :], axis=-1)
    denominator = np.sum(fpd**2 * weight[None, None, :], axis=-1)
    scale_best = numerator / denominator

    pred_dt = fpd * scale_best[:, :, None]
    chi2 = np.zeros(scale_best.shape, dtype=float)
    for i, name in enumerate(fpd_names):
        obs = TIME_DELAY_OBS[name]
        chi2 += split_chi2(pred_dt[:, :, i], obs['mean'], obs['sigma_minus'], obs['sigma_plus'])

    return (
        xr.DataArray(chi2, dims=('chain', 'draw'), coords={'chain': post['chain'], 'draw': post['draw']}),
        xr.DataArray(scale_best, dims=('chain', 'draw'), coords={'chain': post['chain'], 'draw': post['draw']}),
    )

def kappa_ext_chi2(post, fixed_kappa_ext=None):
    obs = KAPPA_EXT_OBS
    if 'kappa_ext' in post:
        value = scalar_da(post['kappa_ext']).values
        coords = {'chain': post['chain'], 'draw': post['draw']}
    elif fixed_kappa_ext is not None:
        template = scalar_da(next(iter(post.data_vars.values())))
        value = np.full((post.sizes['chain'], post.sizes['draw']), float(fixed_kappa_ext), dtype=float)
        coords = {'chain': post['chain'], 'draw': post['draw']}
    else:
        raise KeyError('No kappa_ext in posterior and no fixed_kappa_ext provided.')

    chi2 = split_chi2(value, obs['mean'], obs['sigma_minus'], obs['sigma_plus'])
    return xr.DataArray(chi2, dims=('chain', 'draw'), coords=coords)

def best_from_score(score_da):
    arr = np.asarray(score_da.values, dtype=float)
    flat_index = int(np.nanargmin(arr))
    c, d = np.unravel_index(flat_index, arr.shape)
    return {
        'chain_index': c,
        'draw_index': d,
        'chain_coord': score_da['chain'].values[c].item(),
        'draw_coord': score_da['draw'].values[d].item(),
        'score_best': float(arr[c, d]),
    }


In [ ]:
# ============================================================
# Score one model
# ============================================================
def score_model(cfg, draw_chunk=DRAW_CHUNK):
    print('\n' + '=' * 80)
    print(cfg.name)
    print('=' * 80)

    k_image, k_table = count_model_parameters(cfg.nc_path, cfg.free_vars, cfg.manual_k_offset)
    print('k_image =', k_image)
    display(k_table)

    chi2_img, n_pix = compute_image_chi2(cfg, draw_chunk=draw_chunk)

    post_small = load_small_posterior(cfg.nc_path, ['fpd_31', 'fpd_32', 'fpd_34'] + (['kappa_ext'] if cfg.fixed_kappa_ext is None else []))
    chi2_td, td_scale = time_delay_chi2_with_best_scale(post_small)
    chi2_kappa = kappa_ext_chi2(post_small, fixed_kappa_ext=cfg.fixed_kappa_ext)

    score_image = chi2_img
    score_image_td = chi2_img + chi2_td
    score_full = chi2_img + chi2_td + chi2_kappa

    best_image = best_from_score(score_image)
    best_image_td = best_from_score(score_image_td)
    best_full = best_from_score(score_full)

    k_image_td = k_image + 1  # one fitted time-delay scale, independent of cosmology priors
    k_full = k_image_td      # κ_ext is already counted if it is sampled; fixed κ_ext adds no parameter

    bic_image = best_image['score_best'] + k_image * np.log(n_pix)
    bic_image_td = best_image_td['score_best'] + k_image_td * np.log(n_pix + len(TIME_DELAY_OBS))
    bic_full = best_full['score_best'] + k_full * np.log(n_pix + len(TIME_DELAY_OBS) + 1)

    c_td, d_td = best_image_td['chain_index'], best_image_td['draw_index']
    c_full, d_full = best_full['chain_index'], best_full['draw_index']

    return {
        'model': cfg.name,
        'n_pixel': n_pix,
        'k_image': k_image,
        'k_image_TD': k_image_td,
        'k_full_observable': k_full,
        'chi2_image_best': best_image['score_best'],
        'BIC_image': bic_image,
        'best_image_chain': best_image['chain_coord'],
        'best_image_draw': best_image['draw_coord'],
        'chi2_image_TD_best': best_image_td['score_best'],
        'BIC_image_TD': bic_image_td,
        'best_image_TD_chain': best_image_td['chain_coord'],
        'best_image_TD_draw': best_image_td['draw_coord'],
        'td_scale_best_at_image_TD': float(td_scale.values[c_td, d_td]),
        'chi2_full_observable_best': best_full['score_best'],
        'BIC_full_observable': bic_full,
        'best_full_chain': best_full['chain_coord'],
        'best_full_draw': best_full['draw_coord'],
        'td_scale_best_at_full': float(td_scale.values[c_full, d_full]),
    }


In [ ]:
# ============================================================
# Run BIC comparison
# ============================================================
rows = []
for cfg in MODEL_CONFIGS:
    rows.append(score_model(cfg, draw_chunk=DRAW_CHUNK))

bic_table = pd.DataFrame(rows)
reference = 'EPL'
for col in ['BIC_image', 'BIC_image_TD', 'BIC_full_observable']:
    ref_value = bic_table.loc[bic_table['model'] == reference, col].iloc[0]
    bic_table[f'Delta_{col}_vs_EPL'] = bic_table[col] - ref_value

pd.set_option('display.max_columns', 100)
display(bic_table)

out_csv = PAPER_DIR / 'bic_comparison.csv'
bic_table.to_csv(out_csv, index=False)
print('saved', out_csv)


In [ ]:
# ============================================================
# Plot ΔBIC relative to EPL
# ============================================================
plot_cols = [
    ('Delta_BIC_image_vs_EPL', 'image only'),
    ('Delta_BIC_image_TD_vs_EPL', 'image + time delays'),
    ('Delta_BIC_full_observable_vs_EPL', 'image + time delays + κ_ext'),
]

fig, ax = plt.subplots(figsize=(8.5, 4.2))
x = np.arange(len(bic_table))
width = 0.25
colors = ['0.25', '#356ab7', '#c44e52']

for i, (col, label) in enumerate(plot_cols):
    ax.bar(x + (i - 1) * width, bic_table[col], width=width, label=label, color=colors[i], alpha=0.85)

ax.axhline(0.0, color='k', linewidth=1.0)
ax.set_xticks(x)
ax.set_xticklabels(bic_table['model'], rotation=20, ha='right')
ax.set_ylabel('ΔBIC relative to EPL')
ax.tick_params(axis='both', direction='in')
ax.legend(frameon=False)
fig.tight_layout()

out_pdf = PAPER_DIR / 'bic_comparison_delta.pdf'
fig.savefig(out_pdf, dpi=300, bbox_inches='tight')
print('saved', out_pdf)
plt.show()


## Notes for interpretation

Use ΔBIC, not the absolute BIC, in the paper. Negative ΔBIC means the model is preferred relative to EPL under the chosen score. The image-only score tests image-plane fit quality. The image+time-delay score tests whether the Fermat-potential pattern can match the measured delays after fitting one global time-delay scale. The full-observable score additionally includes the κ_ext constraint. These are BIC-like diagnostics, not exact Bayesian evidences.